# Aggregate all 12 pair results into final master tables

In [ ]:
from pathlib import Path
import importlib
import pandas as pd
import assignment2_gpu_split_module as mod

# Force reload the module so any local .py changes take effect
importlib.reload(mod)

# Hardcoded path to the experiment root—ensures we're pointing to the right GPU node/folder
TRAINING_ROOT = Path(
    "/data/Sajjan_Singh/cv_assign_04/"
    "cv_assign_04_gpu_split_local_fast5/"
    "cv_assign_04_gpu_split_local_fast5/"
    "cv_assign_04"
).resolve()

# Patch the module's internal ROOT so all functions use this specific path
mod.ROOT = TRAINING_ROOT

# Create standard subdirectories if they don't exist (safety step)
for name in ["tables", "plots", "checkpoints", "logs", "reports", "slides", "data", "downloads", "splits", "configs"]:
    (mod.ROOT / name).mkdir(parents=True, exist_ok=True)

# Dataset configs for the local environment
IMAGENET100_MODE = "local_existing"
IMAGENET100_ROOT = mod.ROOT / "data" / "imagenet100"
TRAIN_PRETRAINED = False

print("Patched ROOT:", mod.ROOT)
print("ImageNet root:", IMAGENET100_ROOT)

# --- CHECKPOINT AUDIT ---
# Define the 12 model-dataset-seed combos we expect to find
expected_pairs = [
    "cifar10_vgg_seed42", "cifar10_resnet_seed42", "cifar10_convnext_seed42", "cifar10_vit_seed42",
    "fashion_mnist_vgg_seed42", "fashion_mnist_resnet_seed42", "fashion_mnist_convnext_seed42", "fashion_mnist_vit_seed42",
    "imagenet100_vgg_seed42", "imagenet100_resnet_seed42", "imagenet100_convnext_seed42", "imagenet100_vit_seed42",
]

status_rows = []
for pair in expected_pairs:
    pair_dir = mod.ROOT / "checkpoints" / pair
    # Check for folder existence and the specific CSVs needed for plotting/analysis
    status_rows.append({
        "pair": pair,
        "pair_dir_exists": pair_dir.exists(),
        "pair_all_selectors_csv": (pair_dir / "pair_all_selectors.csv").exists(),
        "selector_summary_csv": (pair_dir / "selector_summary.csv").exists(),
    })

# Render a status table—if anything is 'False', that run needs to be restarted
status_df = pd.DataFrame(status_rows)
display(status_df)

2026-03-21 21:23:29.411057: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-21 21:23:29.460375: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Project root: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04
Device: cuda
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU name: NVIDIA H100 NVL
cifar10              -> https://www.cs.toronto.edu/~kriz/cifar.html
fashion_mnist        -> https://github.com/zalandoresearch/fashion-mnist
imagenet100_hf       -> https://huggingface.co/datasets/clane9/imagenet-100
imagenet100_kaggle   -> https://www.kaggle.com/datasets/ambityga/imagenet100
imagecorruptions     -> https://github.com/bethgelab/imagecorruptions
robustness_repo      -> https://github.com/hendrycks/robustness


- input size: `224`
- policy K values: `1, 3, 5, 10, 15`
- backbones: `vgg16_bn, resnet50, convnext_tiny, vit_base_patch16_224`

Project root: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04
Device: cuda
PyTorch version: 2.6.0+cu124
CUDA available: True
GPU name: NVIDIA H100 NVL
cifar10              -> https://www.cs.toronto.edu/~kriz/cifar.html
fashion_mnist        -> https://github.com/zalandoresearch/fashion-mnist
imagenet100_hf       -> https://huggingface.co/datasets/clane9/imagenet-100
imagenet100_kaggle   -> https://www.kaggle.com/datasets/ambityga/imagenet100
imagecorruptions     -> https://github.com/bethgelab/imagecorruptions
robustness_repo      -> https://github.com/hendrycks/robustness


- input size: `224`
- policy K values: `1, 3, 5, 10, 15`
- backbones: `vgg16_bn, resnet50, convnext_tiny, vit_base_patch16_224`

Patched ROOT: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04
ImageNet root: /data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/data/imagenet100


,pair,pair_dir_exists,pair_all_selectors_csv,selector_summary_csv
0,cifar10_vgg_seed42,True,True,True
1,cifar10_resnet_seed42,True,True,True
2,cifar10_convnext_seed42,True,True,True
3,cifar10_vit_seed42,True,True,True
4,fashion_mnist_vgg_seed42,True,True,True
5,fashion_mnist_resnet_seed42,True,True,True
6,fashion_mnist_convnext_seed42,True,True,True
7,fashion_mnist_vit_seed42,True,True,True
8,imagenet100_vgg_seed42,True,True,True
9,imagenet100_resnet_seed42,True,True,True


In [ ]:
from pathlib import Path
import pandas as pd

# Define where to look (checkpoints) and where to save (tables)
checkpoints_root = mod.ROOT / "checkpoints"
tables_root = mod.ROOT / "tables"
tables_root.mkdir(parents=True, exist_ok=True)

# Files we expect inside every experiment folder
required_files = {
    "master": "pair_all_selectors.csv",
    "clean": "pair_clean_rows.csv",
    "best_corrupt": "pair_best_corrupt_rows.csv",
    "comparison": "pair_clean_vs_best_corrupt_comparison.csv",
}

# Lists to hold dataframes before we concat them
master_parts, clean_parts, best_corrupt_parts, comparison_parts = [], [], [], []
missing_pair_csvs = []

# Loop through every folder in checkpoints
for exp_dir in sorted([p for p in checkpoints_root.iterdir() if p.is_dir()]):
    file_paths = {k: exp_dir / v for k, v in required_files.items()}
    missing_here = [str(path) for path in file_paths.values() if not path.exists()]

    if missing_here:
        # Log folders that didn't finish or are missing data
        missing_pair_csvs.append({"experiment_dir": str(exp_dir), "missing_files": " | ".join(missing_here)})
        continue

    try:
        # Read the 4 main CSVs for this specific experiment
        df_master = pd.read_csv(file_paths["master"])
        df_clean = pd.read_csv(file_paths["clean"])
        df_best = pd.read_csv(file_paths["best_corrupt"])
        df_comp = pd.read_csv(file_paths["comparison"])

        # Tag each row with the folder name so we don't lose track of the source
        for df in (df_master, df_clean, df_best, df_comp):
            if "experiment_dir" not in df.columns:
                df["experiment_dir"] = exp_dir.name

        master_parts.append(df_master)
        clean_parts.append(df_clean)
        best_corrupt_parts.append(df_best)
        comparison_parts.append(df_comp)

    except Exception as e:
        missing_pair_csvs.append({"experiment_dir": str(exp_dir), "missing_files": f"read_error: {e}"})

# Combine everything into giant master DataFrames
master_df = pd.concat(master_parts, ignore_index=True) if master_parts else pd.DataFrame()
clean_rows_df = pd.concat(clean_parts, ignore_index=True) if clean_parts else pd.DataFrame()
best_corrupt_rows_df = pd.concat(best_corrupt_parts, ignore_index=True) if best_corrupt_parts else pd.DataFrame()
comparison_df = pd.concat(comparison_parts, ignore_index=True) if comparison_parts else pd.DataFrame()
missing_pair_csvs_df = pd.DataFrame(missing_pair_csvs)

# --- SAVE EVERYTHING ---
# These are the files you'll actually use for your final tables and plots
master_df.to_csv(tables_root / "full_assignment_all_selectors.csv", index=False)
clean_rows_df.to_csv(tables_root / "full_assignment_clean_rows.csv", index=False)
best_corrupt_rows_df.to_csv(tables_root / "full_assignment_best_corrupt_rows.csv", index=False)
comparison_df.to_csv(tables_root / "clean_vs_best_corrupt_comparison.csv", index=False)
missing_pair_csvs_df.to_csv(tables_root / "aggregation_missing_pair_csvs.csv", index=False)

# Optional: specifically formatted tables for the final report
if not master_df.empty:
    master_df.to_csv(tables_root / "report_all_selector_robustness.csv", index=False)

if not clean_rows_df.empty and not best_corrupt_rows_df.empty:
    clean_rows_df.to_csv(tables_root / "report_clean_accuracy_comparison.csv", index=False)
    best_corrupt_rows_df.to_csv(tables_root / "report_robustness_comparison.csv", index=False)

# Print log of saved files for easy copy-pasting
print("Aggregation complete. Files saved in:", tables_root)

Saved:
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/full_assignment_all_selectors.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/full_assignment_clean_rows.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/full_assignment_best_corrupt_rows.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/clean_vs_best_corrupt_comparison.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/aggregation_missing_pair_csvs.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/report_all_selector_robustness.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_spl

,dataset,model,selector,selector_type,best_epoch,best_val_score,num_val_corruptions,val_corruption_names,val_corruption_severity,test_clean_acc,test_corrupt_mean_acc,robustness_gap,experiment_dir,checkpoint_path
0,cifar10,convnext,clean,clean,24,0.842400,0,NaN,0,0.8356,0.73206,0.10354,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
1,cifar10,convnext,corr_k1_s2,corrupt,18,0.692300,1,gaussian_noise,2,0.8294,0.73342,0.09598,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
2,cifar10,convnext,corr_k3_s2,corrupt,18,0.684600,3,"gaussian_noise, motion_blur, fog",2,0.8294,0.73396,0.09544,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
3,cifar10,convnext,corr_k5_s2,corrupt,24,0.737940,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.8356,0.73138,0.10422,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
4,cifar10,resnet,clean,clean,25,0.663400,0,NaN,0,0.6611,0.44214,0.21896,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
5,cifar10,resnet,corr_k1_s2,corrupt,2,0.262100,1,gaussian_noise,2,0.4208,0.32188,0.09892,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
6,cifar10,resnet,corr_k3_s2,corrupt,24,0.347700,3,"gaussian_noise, motion_blur, fog",2,0.6616,0.44544,0.21616,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
7,cifar10,resnet,corr_k5_s2,corrupt,24,0.446380,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.6616,0.44514,0.21646,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
8,cifar10,vgg,clean,clean,23,0.897100,0,NaN,0,0.8873,0.76452,0.12278,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
9,cifar10,vgg,corr_k1_s2,corrupt,20,0.767800,1,gaussian_noise,2,0.8828,0.76604,0.11676,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...


,dataset,model,selector,selector_type,best_epoch,best_val_score,num_val_corruptions,val_corruption_names,val_corruption_severity,test_clean_acc,test_corrupt_mean_acc,robustness_gap,experiment_dir,checkpoint_path
0,cifar10,convnext,clean,clean,24,0.842400,0,NaN,0,0.835600,0.732060,0.103540,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
1,cifar10,resnet,clean,clean,25,0.663400,0,NaN,0,0.661100,0.442140,0.218960,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
2,cifar10,vgg,clean,clean,23,0.897100,0,NaN,0,0.887300,0.764520,0.122780,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
3,cifar10,vit,clean,clean,25,0.723700,0,NaN,0,0.722100,0.607500,0.114600,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
4,fashion_mnist,convnext,clean,clean,25,0.937250,0,NaN,0,0.929100,0.666980,0.262120,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
5,fashion_mnist,resnet,clean,clean,3,0.695667,0,NaN,0,0.695900,0.397640,0.298260,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
6,fashion_mnist,vgg,clean,clean,24,0.949750,0,NaN,0,0.942900,0.771500,0.171400,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
7,fashion_mnist,vit,clean,clean,25,0.902750,0,NaN,0,0.900100,0.670700,0.229400,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
8,imagenet100,convnext,clean,clean,15,0.548462,0,NaN,0,0.551538,0.346923,0.204615,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
9,imagenet100,resnet,clean,clean,20,0.422265,0,NaN,0,0.420692,0.187385,0.233308,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...


,dataset,model,selector,selector_type,best_epoch,best_val_score,num_val_corruptions,val_corruption_names,val_corruption_severity,test_clean_acc,test_corrupt_mean_acc,robustness_gap,experiment_dir,checkpoint_path
0,cifar10,convnext,corr_k3_s2,corrupt,18,0.684600,3,"gaussian_noise, motion_blur, fog",2,0.829400,0.733960,0.095440,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
1,cifar10,resnet,corr_k3_s2,corrupt,24,0.347700,3,"gaussian_noise, motion_blur, fog",2,0.661600,0.445440,0.216160,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
2,cifar10,vgg,corr_k1_s2,corrupt,20,0.767800,1,gaussian_noise,2,0.882800,0.766040,0.116760,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
3,cifar10,vit,corr_k3_s2,corrupt,25,0.566833,3,"gaussian_noise, motion_blur, fog",2,0.722100,0.608380,0.113720,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
4,fashion_mnist,convnext,corr_k3_s2,corrupt,19,0.716583,3,"gaussian_noise, motion_blur, fog",2,0.926300,0.687580,0.238720,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
5,fashion_mnist,resnet,corr_k5_s2,corrupt,3,0.402133,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.695900,0.397060,0.298840,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
6,fashion_mnist,vgg,corr_k3_s2,corrupt,18,0.783333,3,"gaussian_noise, motion_blur, fog",2,0.940500,0.784060,0.156440,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
7,fashion_mnist,vit,corr_k5_s2,corrupt,18,0.676233,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.884600,0.672500,0.212100,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
8,imagenet100,convnext,corr_k5_s2,corrupt,13,0.350752,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.546385,0.353908,0.192477,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
9,imagenet100,resnet,corr_k5_s2,corrupt,20,0.190615,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.420692,0.187446,0.233246,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...


,dataset,model,selector_clean_selector,selector_type_clean_selector,best_epoch_clean_selector,best_val_score_clean_selector,num_val_corruptions_clean_selector,val_corruption_names_clean_selector,val_corruption_severity_clean_selector,test_clean_acc_clean_selector,...,best_val_score_best_corrupt_selector,num_val_corruptions_best_corrupt_selector,val_corruption_names_best_corrupt_selector,val_corruption_severity_best_corrupt_selector,test_clean_acc_best_corrupt_selector,test_corrupt_mean_acc_best_corrupt_selector,robustness_gap_best_corrupt_selector,experiment_dir_best_corrupt_selector,checkpoint_path_best_corrupt_selector,experiment_dir
0,cifar10,convnext,clean,clean,24,0.842400,0,NaN,0,0.835600,...,0.684600,3,"gaussian_noise, motion_blur, fog",2,0.829400,0.733960,0.095440,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,cifar10_convnext_seed42
1,cifar10,resnet,clean,clean,25,0.663400,0,NaN,0,0.661100,...,0.347700,3,"gaussian_noise, motion_blur, fog",2,0.661600,0.445440,0.216160,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,cifar10_resnet_seed42
2,cifar10,vgg,clean,clean,23,0.897100,0,NaN,0,0.887300,...,0.767800,1,gaussian_noise,2,0.882800,0.766040,0.116760,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,cifar10_vgg_seed42
3,cifar10,vit,clean,clean,25,0.723700,0,NaN,0,0.722100,...,0.566833,3,"gaussian_noise, motion_blur, fog",2,0.722100,0.608380,0.113720,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,cifar10_vit_seed42
4,fashion_mnist,convnext,clean,clean,25,0.937250,0,NaN,0,0.929100,...,0.716583,3,"gaussian_noise, motion_blur, fog",2,0.926300,0.687580,0.238720,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,fashion_mnist_convnext_seed42
5,fashion_mnist,resnet,clean,clean,3,0.695667,0,NaN,0,0.695900,...,0.402133,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.695900,0.397060,0.298840,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,fashion_mnist_resnet_seed42
6,fashion_mnist,vgg,clean,clean,24,0.949750,0,NaN,0,0.942900,...,0.783333,3,"gaussian_noise, motion_blur, fog",2,0.940500,0.784060,0.156440,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,fashion_mnist_vgg_seed42
7,fashion_mnist,vit,clean,clean,25,0.902750,0,NaN,0,0.900100,...,0.676233,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.884600,0.672500,0.212100,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,fashion_mnist_vit_seed42
8,imagenet100,convnext,clean,clean,15,0.548462,0,NaN,0,0.551538,...,0.350752,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.546385,0.353908,0.192477,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,imagenet100_convnext_seed42
9,imagenet100,resnet,clean,clean,20,0.422265,0,NaN,0,0.420692,...,0.190615,5,"gaussian_noise, motion_blur, fog, brightness, ...",2,0.420692,0.187446,0.233246,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...,imagenet100_resnet_seed42


""


In [4]:
all_selector_table, clean_accuracy_table, robustness_table = mod.build_report_tables(
    master_csv=mod.ROOT / "tables" / "full_assignment_all_selectors.csv"
)

print(mod.ROOT / "tables" / "full_assignment_all_selectors.csv")
print(mod.ROOT / "tables" / "full_assignment_clean_rows.csv")
print(mod.ROOT / "tables" / "full_assignment_best_corrupt_rows.csv")
print(mod.ROOT / "tables" / "clean_vs_best_corrupt_comparison.csv")
print(mod.ROOT / "tables" / "report_all_selector_robustness.csv")
print(mod.ROOT / "tables" / "report_clean_accuracy_comparison.csv")
print(mod.ROOT / "tables" / "report_robustness_comparison.csv")

summary_paths = pd.DataFrame({
    "file": [
        "full_assignment_all_selectors.csv",
        "full_assignment_clean_rows.csv",
        "full_assignment_best_corrupt_rows.csv",
        "clean_vs_best_corrupt_comparison.csv",
        "report_all_selector_robustness.csv",
        "report_clean_accuracy_comparison.csv",
        "report_robustness_comparison.csv",
    ],
    "path": [
        str(mod.ROOT / "tables" / "full_assignment_all_selectors.csv"),
        str(mod.ROOT / "tables" / "full_assignment_clean_rows.csv"),
        str(mod.ROOT / "tables" / "full_assignment_best_corrupt_rows.csv"),
        str(mod.ROOT / "tables" / "clean_vs_best_corrupt_comparison.csv"),
        str(mod.ROOT / "tables" / "report_all_selector_robustness.csv"),
        str(mod.ROOT / "tables" / "report_clean_accuracy_comparison.csv"),
        str(mod.ROOT / "tables" / "report_robustness_comparison.csv"),
    ]
})
display(summary_paths)


selector,dataset,model,clean,corr_k1_s2,corr_k3_s2,corr_k5_s2
0,cifar10,convnext,0.732060,0.733420,0.733960,0.731380
1,cifar10,resnet,0.442140,0.321880,0.445440,0.445140
2,cifar10,vgg,0.764520,0.766040,0.764960,0.764640
3,cifar10,vit,0.607500,0.606400,0.608380,0.607520
4,fashion_mnist,convnext,0.666980,0.657720,0.687580,0.686720
5,fashion_mnist,resnet,0.397640,0.341560,0.341060,0.397060
6,fashion_mnist,vgg,0.771500,0.783020,0.784060,0.782680
7,fashion_mnist,vit,0.670700,0.670580,0.670280,0.672500
8,imagenet100,convnext,0.346923,0.302062,0.352538,0.353908
9,imagenet100,resnet,0.187385,0.112323,0.185877,0.187446


,dataset,model,test_clean_acc_clean_selector,test_clean_acc_best_corrupt_selector
0,cifar10,convnext,0.835600,0.829400
1,cifar10,resnet,0.661100,0.661600
2,cifar10,vgg,0.887300,0.882800
3,cifar10,vit,0.722100,0.722100
4,fashion_mnist,convnext,0.929100,0.926300
5,fashion_mnist,resnet,0.695900,0.695900
6,fashion_mnist,vgg,0.942900,0.940500
7,fashion_mnist,vit,0.900100,0.884600
8,imagenet100,convnext,0.551538,0.546385
9,imagenet100,resnet,0.420692,0.420692


,dataset,model,test_corrupt_mean_acc_clean_selector,test_corrupt_mean_acc_best_corrupt_selector,robustness_gap_clean_selector,robustness_gap_best_corrupt_selector
0,cifar10,convnext,0.732060,0.733960,0.103540,0.095440
1,cifar10,resnet,0.442140,0.445440,0.218960,0.216160
2,cifar10,vgg,0.764520,0.766040,0.122780,0.116760
3,cifar10,vit,0.607500,0.608380,0.114600,0.113720
4,fashion_mnist,convnext,0.666980,0.687580,0.262120,0.238720
5,fashion_mnist,resnet,0.397640,0.397060,0.298260,0.298840
6,fashion_mnist,vgg,0.771500,0.784060,0.171400,0.156440
7,fashion_mnist,vit,0.670700,0.672500,0.229400,0.212100
8,imagenet100,convnext,0.346923,0.353908,0.204615,0.192477
9,imagenet100,resnet,0.187385,0.187446,0.233308,0.233246


/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/full_assignment_all_selectors.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/full_assignment_clean_rows.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/full_assignment_best_corrupt_rows.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/clean_vs_best_corrupt_comparison.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/report_all_selector_robustness.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_local_fast5/cv_assign_04_gpu_split_local_fast5/cv_assign_04/tables/report_clean_accuracy_comparison.csv
/data/Sajjan_Singh/cv_assign_04/cv_assign_04_gpu_split_l

,file,path
0,full_assignment_all_selectors.csv,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
1,full_assignment_clean_rows.csv,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
2,full_assignment_best_corrupt_rows.csv,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
3,clean_vs_best_corrupt_comparison.csv,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
4,report_all_selector_robustness.csv,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
5,report_clean_accuracy_comparison.csv,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
6,report_robustness_comparison.csv,/data/Sajjan_Singh/cv_assign_04/cv_assign_04_g...
